# Final Evaluation Orchestrator — *KON-Artist model*
**Project:** Reinforcement Learning Adversarial Audio Attacks against the AASIST3 Detector  
**Author:** Alejandra Lloveras  

This Jupyter Notebook acts as the end-to-end orchestration controller for the final evaluation, executing:
1. **Comparative Benchmark (Ablation Study):** Global evaluation of three frozen model versions (Base, PIK, ACP) against the AASIST3 detector.
2. **Stratified Sub-population Analysis:** Deep-dive into the ACP model (Clusters 0-9) to partition results into "Dominant" vs. "Resilient" acoustic manifolds.

**Engineering Standards:**
- **Resume-ability:** Uses `status_manifest.json` to store run-states and skip already-executed stages in case of kernel restarts.
- **Time-Budget Guard:** Monitors the session elapsed time against the Colab 4-hour budget and exits gracefully before the safety buffer if remaining time is insufficient, avoiding stack trace cell errors.
- **Logging:** Logs evaluation details to `outputs/evaluation/master_benchmark.log`.
- **High-Res Visuals:** Saves publication-ready figures to the evaluation directory.

In [ ]:
# Record Session Start Time
SESSION_START_TIME = time.time()

In [ ]:
# Mount Google Drive for persistent storage when running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

In [ ]:
!git clone -b "eval" https://github.com/MLsound/KON-Artist/

## Cell 1: Environment Setup & Drive Persistence
This cell mounts Google Drive, parses the decoupled parameter settings from `configs/eval_config.yaml`, configures file logging, records `SESSION_START_TIME`, and initializes the `TimeBudgetManager` and `status_manifest.json` tracker.

In [ ]:
# Install dependencies in Google Colab environment to match versions in 05_Colab_training.ipynb
try:
    # import google.colab
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        print("Initializing Google Colab environment with library dependencies...")
        ipy.system("pip install --quiet numpy==1.26.4 scipy==1.13.1 pandas==2.3.3 torch==2.2.2 torchaudio==2.2.2 stable-baselines3==2.4.1 gymnasium==1.0.0 librosa==0.11.0 soundfile==0.13.1 wandb==0.25.1 matplotlib==3.10.8 transformers==4.40.0 datasets==2.19.1 huggingface-hub==0.36.2 pyyaml scikit-learn python-dotenv")
except ImportError:
    pass

import os
import yaml
import json
import logging
import time
import sys
import torch

# Define config path
CONFIG_PATH = "configs/eval_config.yaml"

# 1. Parse configs/eval_config.yaml
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Evaluation configuration file missing at {CONFIG_PATH}!")
    
with open(CONFIG_PATH, 'r') as f:
    eval_config = yaml.safe_load(f)

paths_cfg = eval_config.get("paths", {})
strat_cfg = eval_config.get("stratified_analysis", {})
time_cfg = eval_config.get("time_control", {})
settings_cfg = eval_config.get("settings", {})

MODELS_DIR = paths_cfg.get("weights_dir", "models/final_checkpoints/")
OUTPUTS_DIR = paths_cfg.get("output_base_dir", "outputs/evaluation/")
MANIFEST_PATH = os.path.join(OUTPUTS_DIR, "status_manifest.json")
LOG_PATH = os.path.join(OUTPUTS_DIR, "master_benchmark.log")

# Setup logging to console and file
os.makedirs(OUTPUTS_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(OUTPUTS_DIR, "evaluation_log.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("final_evaluation")
logger.info(f"Master benchmark evaluation session initialized at: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# Ensure model sub-folders for sample storage exist
for folder in ["base_samples", "pik_samples", "acp_samples"]:
    os.makedirs(os.path.join(OUTPUTS_DIR, folder), exist_ok=True)

# Load or initialize status manifest
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, 'r') as f:
        manifest = json.load(f)
    logger.info(f"Loaded existing status manifest: {json.dumps(manifest, indent=2)}")
else:
    manifest = {
        "baseline_evaluated": False,
        "generation_completed": {
            "base": False,
            "pik": False,
            "acp": False
        },
        "scoring_completed": {
            "base": False,
            "pik": False,
            "acp": False
        },
        "visuals_completed": False,
        "task_durations": []
    }
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(manifest, f, indent=4)
    logger.info("Initialized new status manifest.")

# 3. Define the TimeBudgetManager class
class TimeBudgetManager:
    def __init__(self, session_start_time, eval_config, manifest_path):
        self.session_start_time = session_start_time
        self.paths = eval_config.get("paths", {})
        self.stratified_cfg = eval_config.get("stratified_analysis", {})
        self.time_cfg = eval_config.get("time_control", {})
        self.manifest_path = manifest_path
        
        self.outputs_dir = self.paths.get("output_base_dir", "outputs/evaluation/")
        self.session_limit_seconds = self.time_cfg.get("session_limit_hours", 4.0) * 3600
        self.buffer_seconds = self.time_cfg.get("safety_buffer_minutes", 15) * 60
        self.log_path = os.path.join(self.outputs_dir, "master_benchmark.log")
        self.report_path = os.path.join(self.outputs_dir, "final_results.md")
        self.default_duration = self.time_cfg.get("estimated_duration_sec", 1800)

    def get_estimated_task_duration(self, default_duration=None):
        """Calculates rolling average of completed tasks. Returns default_duration if no history."""
        if os.path.exists(self.manifest_path):
            with open(self.manifest_path, 'r') as f:
                manifest_data = json.load(f)
            durations = manifest_data.get("task_durations", [])
            if durations:
                avg = sum(durations) / len(durations)
                logger.info(f"Rolling average task duration: {avg:.2f} seconds based on {len(durations)} tasks.")
                return avg
        logger.info(f"No task history found. Using default estimate: {default_duration} seconds.")
        return default_duration

    def record_task_duration(self, duration):
        """Appends task duration to the manifest tracker."""
        if os.path.exists(self.manifest_path):
            with open(self.manifest_path, 'r') as f:
                manifest_data = json.load(f)
            if "task_durations" not in manifest_data:
                manifest_data["task_durations"] = []
            manifest_data["task_durations"].append(duration)
            with open(self.manifest_path, 'w') as f:
                json.dump(manifest_data, f, indent=4)
            logger.info(f"Recorded task duration: {duration:.2f}s.")

    def should_continue(self, task_duration=None):
        """Checks if there is enough time to execute the next task."""
        if task_duration is None:
            task_duration = self.get_estimated_task_duration()
            
        elapsed = time.time() - self.session_start_time
        time_remaining = (self.session_limit_seconds - elapsed) - self.buffer_seconds
        
        logger.info(f"Time Budget Check: Elapsed = {elapsed/60:.2f}m, Remaining = {time_remaining/60:.2f}m, Required = {task_duration/60:.2f}m")
        
        if time_remaining < task_duration:
            warning_msg = (
                f"RESOURCE WARNING: Time-Budget Guard triggered! "
                f"Remaining time {time_remaining/60:.2f}m is less than required task duration {task_duration/60:.2f}m. "
                f"Initiating graceful shutdown."
            )
            logger.warning(warning_msg)
            
            # Write warning log to master_benchmark.log
            os.makedirs(os.path.dirname(self.log_path), exist_ok=True)
            with open(self.log_path, 'a') as f:
                f.write(f"\n{time.strftime('%Y-%m-%d %H:%M:%S')} [WARNING] {warning_msg}\n")
                
            # Atomically save manifest and export currently gathered results to markdown
            self.finalize_partial_results()
            
            # Raise a dedicated timeout exception instead of using sys.exit() to prevent noisy tracebacks
            raise TimeoutError("STOP_AND_DISCONNECT: " + warning_msg)
        return True

    def finalize_partial_results(self):
        """Generates the results summary with whatever partial results are available."""
        logger.info("Time-Budget Guard: Finalizing results summary before shutdown...")
        
        # Load baseline metrics
        baseline_metrics_path = os.path.join(self.outputs_dir, "baseline_metrics.json")
        baseline_eer = "N/A"
        min_tdcf_val = "N/A"
        if os.path.exists(baseline_metrics_path):
            try:
                with open(baseline_metrics_path, 'r') as f:
                    base_m = json.load(f)
                baseline_eer = f"{base_m['eer'] * 100:.2f}%"
                min_tdcf_val = f"{base_m['min_tdcf']:.4f}"
            except Exception:
                pass
                
        table1 = "| Model Version | Global ASR (%) | Degraded EER (%) | min t-DCF |\n"
        table1 += "| --- | --- | --- | --- |\n"
        table1 += f"| Control Baseline (Clean) | - | {baseline_eer} | {min_tdcf_val} |\n"
        
        models = ["base", "pik", "acp"]
        for m in models:
            m_metrics_path = os.path.join(self.outputs_dir, f"{m}_metrics.json")
            if os.path.exists(m_metrics_path):
                try:
                    with open(m_metrics_path, 'r') as f:
                        metrics = json.load(f)
                    table1 += f"| {m.upper()} | {metrics['asr']:.2f}% | {metrics['eer']*100:.2f}% | {metrics['min_tdcf']:.4f} |\n"
                except Exception:
                    table1 += f"| {m.upper()} | (Partial/Error) | (Partial/Error) | (Partial/Error) |\n"
            else:
                table1 += f"| {m.upper()} | (Not started) | (Not started) | (Not started) |\n"
                
        # Try loading GMM breakdown
        acp_summary_path = os.path.join(self.outputs_dir, "acp_stratified_summary.csv")
        table2 = ""
        table3 = ""
        if os.path.exists(acp_summary_path):
            try:
                import pandas as pd
                df = pd.read_csv(acp_summary_path)
                table2 = "| Cluster ID | Samples | Successful Attacks | Mean Score | ASR (%) | Degraded EER (%) | Acoustic Manifold Group |\n"
                table2 += "| --- | --- | --- | --- | --- | --- | --- |\n"
                for _, row in df.iterrows():
                    table2 += f"| {int(row['cluster_id'])} | {int(row['total_samples'])} | {int(row['successful_attacks'])} | {row['mean_final_score']:.4f} | {row['asr']:.1f}% | {row['eer']:.2f}% | {row['manifold_type']} |\n"
                
                dom_mask = df['manifold_type'] == "Dominant (Vulnerable)"
                res_mask = df['manifold_type'] == "Resilient (Robust)"
                mean_asr_dom = df[dom_mask]['asr'].mean() if sum(dom_mask) > 0 else 0.0
                mean_eer_dom = df[dom_mask]['eer'].mean() if sum(dom_mask) > 0 else 0.0
                mean_asr_res = df[res_mask]['asr'].mean() if sum(res_mask) > 0 else 0.0
                mean_eer_res = df[res_mask]['eer'].mean() if sum(res_mask) > 0 else 0.0
                
                table3 = "| Manifold Segment | Cluster Count | Average ASR (%) | Average EER (%) | Description |\n"
                table3 += "| --- | --- | --- | --- | --- |\n"
                table3 += f"| Dominant (Vulnerable) | {sum(dom_mask)} | {mean_asr_dom:.2f}% | {mean_eer_dom:.2f}% | High attack vulnerability; RL agent easily penetrates detector defenses. |\n"
                table3 += f"| Resilient (Robust) | {sum(res_mask)} | {mean_asr_res:.2f}% | {mean_eer_res:.2f}% | Hardened manifolds; high resistance to adversarial signal deformation. |\n"
            except Exception as e:
                logger.error(f"Error compiling stratified table for partial results: {e}")
                
        report_content = f"""# Final Evaluation: Results Summary (Time-Budget Terminated)
Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}

> [!WARNING]
> This run was terminated early by the Time-Budget Guard to protect the GPU budget. Below are the partial or completed metrics captured before shutdown.

## 1. Global Ablation Study (Table 1)
{table1}
"""
        if table2:
            report_content += f"""
## 2. ACP Acoustic Manifold Breakdown (Table 2)
{table2}

## 3. Macro Acoustic Manifold Analysis (Table 3)
{table3}
"""
        
        os.makedirs(os.path.dirname(self.report_path), exist_ok=True)
        with open(self.report_path, 'w') as f:
            f.write(report_content)
        logger.info(f"Saved partial report to {self.report_path}")

# 4. Instantiate Time Budget Manager
time_budget_manager = TimeBudgetManager(SESSION_START_TIME, eval_config, MANIFEST_PATH)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
logger.info(f"TimeBudgetManager initialized. Active device: {device}")

## Cell 2: Stage 1: Control Baseline
To establish a clean reference, we load a balanced subset of un-attacked genuine (Bonafide) and synthetic (Spoof) samples from the ASVspoof 2019 validation split using `get_asvspoof_loader`. We then save the control dataset locally and run `baseline_evaluate.py` to calculate the baseline detector's EER and min t-DCF, which are exported to `outputs/evaluation/baseline_metrics.json`.

In [ ]:
import io
import torchaudio
from src.data.loader import get_asvspoof_loader
from scripts.baseline_evaluate import evaluate as run_baseline_evaluate

raw_samples_dir = paths_cfg.get("raw_data_dir", "data/asvspoof_2019/test/wav/")
raw_protocol_path = paths_cfg.get("protocol_file", "data/asvspoof_2019/test/protocol.txt")

# 1. Download and compile control baseline subset from HF if not present
if not os.path.exists(raw_samples_dir) or not os.path.exists(raw_protocol_path):
    os.makedirs(raw_samples_dir, exist_ok=True)
    logger.info("Downloading/streaming ASVspoof 2019 validation samples...")
    ds = get_asvspoof_loader(split="validation", seed=42)
    
    bonafide_count = 0
    spoof_count = 0
    num_eval_samples = settings_cfg.get("num_eval_samples", 50)
    target_count_per_class = num_eval_samples // 2  # Total balanced size = num_eval_samples
    protocol_lines = []
    
    for sample in ds:
        original_label = sample["key"]
        mapped_label = 1 - original_label  # 0 (bonafide) -> 1, 1 (spoof) -> 0
        
        if mapped_label == 1:
            if bonafide_count >= target_count_per_class:
                continue
            label_str = "bonafide"
            bonafide_count += 1
        else:
            if spoof_count >= target_count_per_class:
                continue
            label_str = "spoof"
            spoof_count += 1
            
        file_id = os.path.basename(sample["path"].split(".")[0]) if "path" in sample else f"sample_{bonafide_count + spoof_count:04d}"
        audio_bytes = sample["audio"]["bytes"]
        waveform, sr = torchaudio.load(io.BytesIO(audio_bytes))
        
        # Save clean wave file
        wav_path = os.path.join(raw_samples_dir, f"{file_id}.wav")
        torchaudio.save(wav_path, waveform, sr)
        
        # Standard ASVspoof protocol format line
        protocol_lines.append(f"LA_0000 {file_id} - - {label_str}\n")
        
        if bonafide_count >= target_count_per_class and spoof_count >= target_count_per_class:
            break
            
    with open(raw_protocol_path, 'w') as f:
        f.writelines(protocol_lines)
    logger.info(f"Control baseline dataset compiled: {bonafide_count} bonafide & {spoof_count} spoof samples saved.")

# 2. Run baseline evaluation if not already completed (checks Time-Budget Guard)
try:
    if not manifest.get("baseline_evaluated", False):
        # Check time budget (baseline download/eval estimates e.g. 300 seconds)
        time_budget_manager.should_continue(task_duration=300)
        
        start_t = time.time()
        logger.info("Evaluating baseline detector performance...")
        eer, min_tdcf = run_baseline_evaluate(
            model_path="MTUCI/AASIST3",
            data_dir=raw_samples_dir,
            protocol_path=raw_protocol_path,
            scores_out=os.path.join(OUTPUTS_DIR, "baseline_scores.txt"),
            device=device
        )
        
        duration = time.time() - start_t
        time_budget_manager.record_task_duration(duration)
        
        baseline_metrics = {
            "eer": float(eer),
            "min_tdcf": float(min_tdcf),
            "bonafide_samples": target_count_per_class,
            "spoof_samples": target_count_per_class
        }
        baseline_metrics_path = os.path.join(OUTPUTS_DIR, "baseline_metrics.json")
        with open(baseline_metrics_path, 'w') as f:
            json.dump(baseline_metrics, f, indent=4)
            
        manifest["baseline_evaluated"] = True
        with open(MANIFEST_PATH, 'w') as f:
            json.dump(manifest, f, indent=4)
        logger.info(f"Baseline Evaluation Finished. EER: {eer*100:.4f}%, min t-DCF: {min_tdcf:.4f}")
    else:
        logger.info("Control baseline evaluation already completed. Skipping.")
except TimeoutError as e:
    logger.warning(f"Execution Halted Gracefully via Time-Budget Guard: {e}")
    SHOULD_DISCONNECT_RUNTIME = True

## Cell 3: Stage 2: Iterative Generation with Idempotency
Loops through the three model checkpoints (Base, PIK, and ACP). For each model, it checks the status manifest to skip already generated sets. It calls `should_continue` on the `TimeBudgetManager` (using rolling average task duration) before running the generation.  
**ACP Specifics:** The script executes with `--model_type acp` which triggers loading the GMM model (path loaded from config) to predict baseline GMM cluster assignments (0-9) for each sample, producing `outputs/evaluation/acp_samples/cluster_map.csv`.

In [ ]:
from scripts.generate_attack_samples import run_generation

# Define model checkpoints and output setups based on configs - No fallbacks, strict error throwing
models_cfg = eval_config.get("models", {})
models_config = {}
for model_name, m_info in models_cfg.items():
    models_config[model_name] = {
        "path": os.path.join(MODELS_DIR, m_info.get("filename")),
        "out_dir": os.path.join(OUTPUTS_DIR, m_info.get("sample_dir")),
        "model_type": m_info.get("model_type", model_name)
    }

num_samples_to_generate = settings_cfg.get("num_eval_samples", 50) // 2  # Balanced spoof count

try:
    for model_name, cfg in models_config.items():
        if not manifest["generation_completed"].get(model_name, False):
            if not os.path.exists(cfg["path"]):
                raise FileNotFoundError(f"Trained agent checkpoint for {model_name} not found at {cfg['path']}!")
                
            # Check time budget using rolling average of task duration
            time_budget_manager.should_continue()
            
            logger.info(f"Running adversarial sample generation for {model_name}...")
            start_t = time.time()
            run_generation(
                model_path=cfg["path"],
                num_samples=num_samples_to_generate,
                output_dir=cfg["out_dir"],
                model_type=cfg["model_type"]
            )
            duration = time.time() - start_t
            time_budget_manager.record_task_duration(duration)
            
            manifest["generation_completed"][model_name] = True
            with open(MANIFEST_PATH, 'w') as f:
                json.dump(manifest, f, indent=4)
            logger.info(f"Adversarial samples generated successfully for model: {model_name}.")
        else:
            logger.info(f"Adversarial samples for {model_name} already exist in manifest. Skipping generation.")
except TimeoutError as e:
    logger.warning(f"Execution Halted Gracefully via Time-Budget Guard: {e}")
    SHOULD_DISCONNECT_RUNTIME = True

## Cell 4: Stage 3: Protocol Scoring & Stratified Analysis
This cell computes global metrics (ASR and Degraded EER) for all models, utilizing `should_continue` checks before execution.  
**Stratified Scoring:** For the ACP model, it ingests the `cluster_map.csv` predictions, reads the generated sample scores, and aggregates ASR and degraded EER grouped by Cluster ID (0-9) using scikit-learn metrics to compare Dominant vs. Resilient acoustic manifolds.

In [ ]:
import pandas as pd
import shutil
import numpy as np
from scripts.baseline_evaluate import evaluate as run_baseline_evaluate
from src.utils.metrics import compute_eer

global_results = {}

try:
    # 1. Scoring & EER/ASR Calculations per Model
    for model_name, cfg in models_config.items():
        if not os.path.exists(cfg["path"]):
            continue

        combined_wav_dir = os.path.join(OUTPUTS_DIR, f"{model_name}_eval_wav")
        combined_protocol = os.path.join(OUTPUTS_DIR, f"{model_name}_eval_protocol.txt")
        
        if not manifest["scoring_completed"].get(model_name, False):
            # Check time budget (scoring task estimates e.g. 120 seconds)
            time_budget_manager.should_continue(task_duration=120)
            
            logger.info(f"Compiling evaluation dataset for scoring {model_name}...")
            start_t = time.time()
            os.makedirs(combined_wav_dir, exist_ok=True)
            protocol_lines = []
            
            # Copy baseline bonafide signals
            with open(raw_protocol_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    file_id, label = parts[1], parts[4]
                    if label == 'bonafide':
                        shutil.copy2(os.path.join(raw_samples_dir, f"{file_id}.wav"), os.path.join(combined_wav_dir, f"{file_id}.wav"))
                        protocol_lines.append(line)
            
            # Copy attacked spoof signals
            attack_files = [f for f in os.listdir(cfg["out_dir"]) if f.endswith('.wav')]
            for filename in attack_files:
                file_id = os.path.splitext(filename)[0]
                shutil.copy2(os.path.join(cfg["out_dir"], filename), os.path.join(combined_wav_dir, filename))
                protocol_lines.append(f"LA_0000 {file_id} - - spoof\n")
                
            with open(combined_protocol, 'w') as f:
                f.writelines(protocol_lines)
                
            # Run detector scoring to get Degraded EER
            logger.info(f"Scoring AASIST3 detector robustness on {model_name} adversarial samples...")
            eer, min_tdcf = run_baseline_evaluate(
                model_path="MTUCI/AASIST3",
                data_dir=combined_wav_dir,
                protocol_path=combined_protocol,
                scores_out=os.path.join(OUTPUTS_DIR, f"{model_name}_scores.txt"),
                device=device
            )
            
            # Ingest generation metadata for Attack Success Rate (ASR)
            metadata_path = os.path.join(cfg["out_dir"], "generation_metadata.json")
            with open(metadata_path, 'r') as f:
                meta = json.load(f)
            total_spoofs = len(meta)
            successful_spoofs = sum(1 for m in meta if m["success"])
            asr = (successful_spoofs / total_spoofs) * 100 if total_spoofs > 0 else 0.0
            
            model_metrics = {
                "eer": float(eer),
                "min_tdcf": float(min_tdcf),
                "asr": float(asr),
                "total_samples": len(protocol_lines)
            }
            
            with open(os.path.join(OUTPUTS_DIR, f"{model_name}_metrics.json"), 'w') as f:
                json.dump(model_metrics, f, indent=4)
                
            duration = time.time() - start_t
            time_budget_manager.record_task_duration(duration)
            
            manifest["scoring_completed"][model_name] = True
            with open(MANIFEST_PATH, 'w') as f:
                json.dump(manifest, f, indent=4)
            logger.info(f"{model_name} Scoring complete: EER: {eer*100:.4f}%, ASR: {asr:.2f}%")
        else:
            logger.info(f"{model_name} scoring metrics already computed. Loading cached data...")
            with open(os.path.join(OUTPUTS_DIR, f"{model_name}_metrics.json"), 'r') as f:
                model_metrics = json.load(f)
                
        global_results[model_name] = model_metrics

    # 2. Stratified Sub-population Analysis (ACP Model Only)
    logger.info("Conducting stratified acoustic manifold analysis for ACP...")
    acp_meta_path = os.path.join(OUTPUTS_DIR, "acp_samples", "generation_metadata.json")
    acp_cmap_path = os.path.join(OUTPUTS_DIR, "acp_samples", "cluster_map.csv")
    acp_scores_path = os.path.join(OUTPUTS_DIR, "acp_scores.txt")

    if os.path.exists(acp_meta_path) and os.path.exists(acp_cmap_path) and os.path.exists(acp_scores_path):
        df_meta = pd.read_json(acp_meta_path)
        df_cmap = pd.read_csv(acp_cmap_path)
        
        # Load evaluation scores
        scores_dict = {}
        with open(acp_scores_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    scores_dict[parts[0]] = float(parts[1])
                    
        # Align files and merge dataframes - Using secure splitext
        df_meta['file_id'] = df_meta['filename'].apply(lambda x: os.path.splitext(x)[0])
        df_cmap['file_id'] = df_cmap['filename'].apply(lambda x: os.path.splitext(x)[0])
        df_merged = pd.merge(df_meta, df_cmap, on='file_id')
        df_merged['detector_score'] = df_merged['file_id'].map(scores_dict)
        
        # Group results per cluster
        cluster_summary = df_merged.groupby('cluster_id').agg(
            total_samples=('success', 'count'),
            successful_attacks=('success', 'sum'),
            mean_final_score=('final_score', 'mean')
        ).reset_index()
        
        cluster_summary['asr'] = (cluster_summary['successful_attacks'] / cluster_summary['total_samples']) * 100
        
        # Ingest baseline scores using the protocol file to map file_id to its label
        label_map = {}
        with open(raw_protocol_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    label_map[parts[1]] = parts[4]
                    
        # Load baseline scores mapping
        baseline_scores_dict = {}
        with open(os.path.join(OUTPUTS_DIR, "baseline_scores.txt"), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    baseline_scores_dict[parts[0]] = float(parts[1])
                    
        # Pull bonafide scores strictly matching label
        bonafide_scores = [score for fid, score in baseline_scores_dict.items() if label_map.get(fid) == 'bonafide']
                    
        cluster_eers = {}
        for cid in range(strat_cfg.get("cluster_count", 10)):
            c_spoofs = df_merged[df_merged['cluster_id'] == cid]['detector_score'].dropna().values
            if len(c_spoofs) > 0 and len(bonafide_scores) > 0:
                eer_val, _, _, _ = compute_eer(np.array(bonafide_scores), c_spoofs)
                cluster_eers[cid] = float(eer_val) * 100
            else:
                cluster_eers[cid] = 0.0
                
        cluster_summary['eer'] = cluster_summary['cluster_id'].map(cluster_eers)
        
        # Segment manifolds based on configuration thresholds
        dom_threshold = strat_cfg.get("dominant_asr_threshold", 0.40) * 100.0
        res_threshold = strat_cfg.get("resilient_asr_threshold", 0.10) * 100.0
        def segment_manifold(row):
            if row['asr'] >= dom_threshold:
                return "Dominant (Vulnerable)"
            elif row['asr'] <= res_threshold:
                return "Resilient (Robust)"
            else:
                return "Intermediate"
            
        cluster_summary['manifold_type'] = cluster_summary.apply(segment_manifold, axis=1)
        
        cluster_summary.to_csv(os.path.join(OUTPUTS_DIR, "acp_stratified_summary.csv"), index=False)
        logger.info("Stratified analysis results written.")
        print("\nACP STRATIFIED ACOUSTIC MANIFOLD SCORES:")
        print(cluster_summary.to_string(index=False))
    else:
        logger.warning("Failed to run stratified analysis: Missing CSV or score files.")
except TimeoutError as e:
    logger.warning(f"Execution Halted Gracefully via Time-Budget Guard: {e}")
    SHOULD_DISCONNECT_RUNTIME = True

## Cell 5: Stage 4: Visual Reporting & Summary
Generates final figures and saves them in the outputs folder. Then, compiles a comprehensive report containing global benchmarks and cluster breakdowns, and exports it to `outputs/evaluation/final_results.md`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Apply styling schema
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.titlesize': 17
})

# Load baseline metrics
baseline_metrics_path = os.path.join(OUTPUTS_DIR, "baseline_metrics.json")
if os.path.exists(baseline_metrics_path):
    with open(baseline_metrics_path, 'r') as f:
        base_m = json.load(f)
    baseline_eer = base_m["eer"] * 100
else:
    baseline_eer = 0.0

# Plot 1: Ablation Study Comparison
models = list(global_results.keys())
asr_vals = [global_results[m]["asr"] for m in models]
eer_vals = [global_results[m]["eer"] * 100 for m in models]

fig, ax1 = plt.subplots(figsize=(10, 6))
color = '#1f77b4'
ax1.set_xlabel('RL Model Checkpoint (Agent)', fontweight='bold', labelpad=10)
ax1.set_ylabel('Attack Success Rate (ASR %)', color=color, fontweight='bold', labelpad=10)
bars1 = ax1.bar(np.arange(len(models)) - 0.2, asr_vals, width=0.4, color=color, alpha=0.85, label='ASR')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(np.arange(len(models)))
ax1.set_xticklabels([m.upper() for m in models], fontweight='bold')

for bar in bars1:
    h = bar.get_height()
    ax1.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('Degraded EER (%)', color=color, fontweight='bold', labelpad=10)
bars2 = ax2.bar(np.arange(len(models)) + 0.2, eer_vals, width=0.4, color=color, alpha=0.85, label='Degraded EER')
ax2.tick_params(axis='y', labelcolor=color)

# Horizontal reference line for clean baseline detector EER
ax2.axhline(y=baseline_eer, color='black', linestyle='--', linewidth=1.5, label=f'Un-attacked Baseline EER ({baseline_eer:.2f}%)')

for bar in bars2:
    h = bar.get_height()
    ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

plt.title('Ablation Study: Efficacy vs Detector EER Degradation', fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "ablation_comparison.png"), dpi=300)
plt.show()

# Plot 2: Cluster Vulnerability Bar Chart
acp_summary_path = os.path.join(OUTPUTS_DIR, "acp_stratified_summary.csv")
if os.path.exists(acp_summary_path):
    df_cluster = pd.read_csv(acp_summary_path)
    
    # Align all clusters 0-9
    df_cluster = pd.merge(pd.DataFrame({'cluster_id': range(strat_cfg.get("cluster_count", 10))}), df_cluster, on='cluster_id', how='left')
    df_cluster[['total_samples', 'successful_attacks', 'mean_final_score', 'asr', 'eer']] = df_cluster[['total_samples', 'successful_attacks', 'mean_final_score', 'asr', 'eer']].fillna(0)
    df_cluster['manifold_type'] = df_cluster['manifold_type'].fillna("No Data")
    
    fig, ax1 = plt.subplots(figsize=(11, 6))
    color = '#1f77b4'
    ax1.set_xlabel('Acoustic Cluster (predicted GMM ID)', fontweight='bold', labelpad=10)
    ax1.set_ylabel('ASR (%)', color=color, fontweight='bold', labelpad=10)
    bars_c = ax1.bar(df_cluster['cluster_id'] - 0.2, df_cluster['asr'], width=0.4, color=color, alpha=0.8, label='ASR')
    ax1.set_xticks(range(strat_cfg.get("cluster_count", 10)))
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()
    color = '#e377c2'
    ax2.set_ylabel('Degraded EER (%)', color=color, fontweight='bold', labelpad=10)
    bars_e = ax2.bar(df_cluster['cluster_id'] + 0.2, df_cluster['eer'], width=0.4, color=color, alpha=0.8, label='EER')
    ax2.tick_params(axis='y', labelcolor=color)
    
    for bar in bars_c:
        h = bar.get_height()
        if h > 0:
            ax1.annotate(f'{h:.0f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 2), textcoords="offset points", ha='center', va='bottom', fontsize=9)
            
    for bar in bars_e:
        h = bar.get_height()
        if h > 0:
            ax2.annotate(f'{h:.0f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 2), textcoords="offset points", ha='center', va='bottom', fontsize=9)
            
    plt.title('ACP Model Stratified Acoustic Cluster Vulnerability', fontweight='bold', pad=15)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "acp_cluster_vulnerability.png"), dpi=300)
    plt.show()
    
    # Generate structured Markdown tables
    # Table 1: Ablation Study
    table1 = "| Model Version | Global ASR (%) | Degraded EER (%) | min t-DCF |\n"
    table1 += "| --- | --- | --- | --- |\n"
    table1 += f"| Control Baseline (Clean) | - | {baseline_eer:.2f}% | {base_m['min_tdcf']:.4f} |\n"
    for m in models:
        table1 += f"| {m.upper()} | {global_results[m]['asr']:.2f}% | {global_results[m]['eer']*100:.2f}% | {global_results[m]['min_tdcf']:.4f} |\n"
        
    # Table 2: Cluster Breakdown
    table2 = "| Cluster ID | Samples | Successful Attacks | Mean Score | ASR (%) | Degraded EER (%) | Acoustic Manifold Group |\n"
    table2 += "| --- | --- | --- | --- | --- | --- | --- |\n"
    for _, row in df_cluster.iterrows():
        table2 += f"| {int(row['cluster_id'])} | {int(row['total_samples'])} | {int(row['successful_attacks'])} | {row['mean_final_score']:.4f} | {row['asr']:.1f}% | {row['eer']:.2f}% | {row['manifold_type']} |\n"
        
    # Table 3: Macro Analysis
    dom_mask = df_cluster['manifold_type'] == "Dominant (Vulnerable)"
    res_mask = df_cluster['manifold_type'] == "Resilient (Robust)"
    
    mean_asr_dom = df_cluster[dom_mask]['asr'].mean() if sum(dom_mask) > 0 else 0.0
    mean_eer_dom = df_cluster[dom_mask]['eer'].mean() if sum(dom_mask) > 0 else 0.0
    mean_asr_res = df_cluster[res_mask]['asr'].mean() if sum(res_mask) > 0 else 0.0
    mean_eer_res = df_cluster[res_mask]['eer'].mean() if sum(res_mask) > 0 else 0.0
    
    table3 = "| Manifold Segment | Cluster Count | Average ASR (%) | Average EER (%) | Description |\n"
    table3 += "| --- | --- | --- | --- | --- |\n"
    table3 += f"| Dominant (Vulnerable) | {sum(dom_mask)} | {mean_asr_dom:.2f}% | {mean_eer_dom:.2f}% | High attack vulnerability; RL agent easily penetrates detector defenses. |\n"
    table3 += f"| Resilient (Robust) | {sum(res_mask)} | {mean_asr_res:.2f}% | {mean_eer_res:.2f}% | Hardened manifolds; high resistance to adversarial signal deformation. |\n"
    
    final_report = f"""# Final Evaluation: Results Summary
Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

## 1. Global Ablation Study (Table 1)
Evaluates the performance of un-attacked control samples and the adversarial attacks generated by all three RL agent versions.
{table1}

## 2. ACP Acoustic Manifold Breakdown (Table 2)
Detailed scoring breakdown across the 10 acoustic partitions for the ACP agent.
{table2}

## 3. Macro Acoustic Manifold Analysis (Table 3)
Segmentation of acoustic space based on ASR vulnerability boundary levels.
{table3}
"""
    
    report_path = os.path.join(OUTPUTS_DIR, "final_results.md")
    with open(report_path, 'w') as f:
        f.write(final_report)
        
    manifest["visuals_completed"] = True
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(manifest, f, indent=4)
        
    print("\n" + "="*80)
    print("RESULTS REPORT SUMMARY (EXPORTED TO " + report_path + ")")
    print("="*80)
    print(final_report)
    print("="*80)
else:
    logger.error("Acoustic cluster summary file missing. Final visual reporting skipped.")

## Cell 6: Persistent Resource Teardown
This cell executes a clean environment tear-down by unassigning the runtime from the active hardware backend to preserve computational resource credits, only if an early exit was flagged by the Time-Budget Guard.

In [ ]:
if 'SHOULD_DISCONNECT_RUNTIME' in locals() and SHOULD_DISCONNECT_RUNTIME:
    print("All partial results successfully written to Google Drive.")
    print("Disconnecting VM runtime allocation to save compute credits...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except ImportError:
        print("Not running in Google Colab environment. Disconnect skipped.")
else:
    print("Evaluation completed successfully within the allocated session budget. Keeping runtime alive.")